# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [3]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** **Line chart, indexed to 1970 = 100.** A line shows change over time, and indexing every continent to its own 1970 size answers the *relative* growth question directly: raw totals would let Asia's huge population swamp everything else. Africa is the one accent colour (it is the answer), the other continents are grey context, and direct end-of-line labels replace the legend.

In [4]:
# Q1
ACCENT, GREY = '#D55E00', '#A6A6A6'

cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100      # 1970 = 100

end = cont[cont['Year'] == 2022].set_index('Continent')['Index']
fastest = end.idxmax()

fig = px.line(
    cont, x='Year', y='Index', color='Continent', markers=True,
    color_discrete_map={c: (ACCENT if c == fastest else GREY) for c in end.index},
    template='plotly_white',
    title=f"{fastest}'s population has nearly quadrupled since 1970 ({end[fastest]:.0f}% of its 1970 size),<br>"
          f"far faster than any other continent",
)
fig.update_traces(line_width=2, marker_size=5)
fig.update_traces(line_width=4, marker_size=7, selector=dict(name=fastest))

# direct labels at the end of each line (small vertical nudges stop the grey labels overlapping)
nudge = {'Oceania': 12, 'South America': 0, 'Asia': -12}
for name, val in end.items():
    fig.add_annotation(x=2022, y=val, text=f"<b>{name}</b> {val:.0f}" if name == fastest else f"{name} {val:.0f}",
                       showarrow=False, xanchor='left', xshift=8, yshift=nudge.get(name, 0),
                       font=dict(color=ACCENT if name == fastest else '#666666', size=12))

fig.add_hline(y=100, line_dash='dot', line_color='#BBBBBB')
fig.update_layout(showlegend=False, title_x=0, title_font_size=18, margin=dict(r=130, t=90),
                  yaxis_title='Population relative to 1970 (1970 = 100)', xaxis_title=None)
fig.update_xaxes(range=[1968, 2036], tickvals=[1970, 1980, 1990, 2000, 2010, 2020], showgrid=False)
fig.update_yaxes(showgrid=False, range=[80, 420])
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** **Sorted horizontal bar chart.** The question is a ranking of 15 named countries by one number, so bars sorted from largest to smallest make the order and the size gaps instant to read, and horizontal bars give the country names room. Africa is the single accent colour and everything else is grey, so the "how many are African?" part can be counted at a glance; value labels on the bars replace the x-axis.

In [5]:
# Q2
ACCENT, GREY = '#D55E00', '#A6A6A6'

gain = df.assign(Gain=df['2022 Population'] - df['1970 Population'])
top15 = gain.nlargest(15, 'Gain').sort_values('Gain')            # ascending: biggest ends up on top
top15['Region'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
top15['Label'] = (top15['Gain'] / 1e6).round().astype(int).astype(str) + 'M'

n_africa = (top15['Continent'] == 'Africa').sum()
top_african = top15[top15['Continent'] == 'Africa'].iloc[-1]

fig = px.bar(
    top15, x='Gain', y='Country', orientation='h', color='Region', text='Label',
    color_discrete_map={'Africa': ACCENT, 'Other continents': GREY},
    template='plotly_white',
    title=f"Only {n_africa} of the 15 countries that added the most people since 1970 are African,<br>"
          f"led by {top_african['Country']} (+{top_african['Gain'] / 1e6:.0f}M)",
)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_yaxes(categoryorder='array', categoryarray=top15['Country'].tolist(), title=None)
fig.update_xaxes(visible=False)
fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=90, r=40), legend_title_text='',
                  legend=dict(orientation='h', yanchor='bottom', y=0.02, xanchor='right', x=1))
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** **Scatter plot on log-log axes.** Two numeric variables and a relationship question call for a scatter plot. Both variables span many orders of magnitude (Monaco to Russia), so log scales are needed to see any pattern; colour encodes continent so the continent differences show up, and a dashed fitted line gives the "expected" population for a given area so that the outliers I label are visibly off the pattern.

In [6]:
# Q3
CONT_COLORS = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
               'North America': '#E69F00', 'South America': '#56B4E9', 'Oceania': '#666666'}  # Okabe-Ito, CVD-safe, no red/green pair

d = df.copy()
lx, ly = np.log10(d['Area']), np.log10(d['2022 Population'])
slope, intercept = np.polyfit(lx, ly, 1)                          # fit in log-log space
d['Label'] = np.where(d['Country'].isin(
    ['Greenland', 'Macau', 'Singapore', 'Bangladesh', 'Canada', 'Australia', 'Russia', 'China', 'India']),
    d['Country'], '')

fig = px.scatter(
    d, x='Area', y='2022 Population', color='Continent', text='Label', hover_name='Country',
    log_x=True, log_y=True, color_discrete_map=CONT_COLORS, template='plotly_white', opacity=0.8,
    title="Bigger countries tend to have more people: Asia sits above the trend, Oceania and<br>"
          "South America below, and Greenland is the most extreme outlier",
)
fig.update_traces(marker_size=8, textposition='top center', textfont_size=11)

xs = np.array([lx.min(), lx.max()])
fig.add_scatter(x=10 ** xs, y=10 ** (intercept + slope * xs), mode='lines',
                line=dict(color='#999999', dash='dash', width=1.5), hoverinfo='skip', showlegend=False)

fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=90), legend_title_text='',
                  legend=dict(orientation='h', yanchor='bottom', y=1.0, xanchor='left', x=0),
                  xaxis_title='Land area, km² (log scale)', yaxis_title='2022 population (log scale)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** **Box plot on a log axis (with every country shown as a dot).** Box plots summarise a distribution per category (median, quartiles, outliers), so they show the spread within continents and let me compare it across continents; the dots keep individual countries visible. Density ranges from under 1 to over 20,000 people/km², so a log axis is required, and continents are sorted by median so the across-continent ranking reads left to right.

In [7]:
# Q4
CONT_COLORS = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
               'North America': '#E69F00', 'South America': '#56B4E9', 'Oceania': '#666666'}

order = df.groupby('Continent')['Density'].median().sort_values().index.tolist()
eu = df.loc[df['Continent'] == 'Europe', 'Density']

fig = px.box(
    df, x='Continent', y='Density', color='Continent', points='all', log_y=True, hover_name='Country',
    category_orders={'Continent': order}, color_discrete_map=CONT_COLORS, template='plotly_white',
    title=f"Density differs more within continents than between them:<br>"
          f"Europe alone runs from {eu.min():.0f} to {eu.max():,.0f} people per km²",
)
fig.update_traces(marker_size=4, marker_opacity=0.6, line_width=1.5)

# label the densest country on each continent
for cont_name, idx in df.groupby('Continent')['Density'].idxmax().items():
    fig.add_annotation(x=cont_name, y=np.log10(df.loc[idx, 'Density']),            # log axis -> annotate in log10 units
                       text=f"{df.loc[idx, 'Country']} {df.loc[idx, 'Density']:,.0f}",
                       showarrow=False, xanchor='left', xshift=10, font=dict(size=10, color='#555555'))

fig.update_layout(showlegend=False, title_x=0, title_font_size=18, margin=dict(t=90),
                  xaxis_title=None, yaxis_title='People per km², 2022 (log scale)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** **Choropleth map with a diverging colour scale.** The question has a geographic part ("where in the world") and a growth/shrink part, so a map coloured by % change fits best. A diverging orange-purple scale (CVD-safe, no red/green) is centred on 0 so shrinking and growing countries are opposite hues; the scale is capped at ±60% so a few extreme growers (Qatar, UAE) don't wash out the rest, and hover still shows exact values.

In [8]:
# Q5
d = df.copy()
d['Change'] = (d['2022 Population'] / d['2000 Population'] - 1) * 100       # % change 2000 -> 2022
d['Change (capped)'] = d['Change'].clip(-60, 60)

n_grew, n_shrank = int((d['Change'] > 0).sum()), int((d['Change'] < 0).sum())
n_eu = int(((d['Change'] < 0) & (d['Continent'] == 'Europe')).sum())

fig = px.choropleth(
    d, locations='CCA3', locationmode='ISO-3', color='Change (capped)', hover_name='Country',
    hover_data={'Change': ':.1f', 'Change (capped)': False, 'CCA3': False},
    color_continuous_scale='PuOr',           # low = orange (shrank), high = purple (grew); CVD-safe
    range_color=(-60, 60), template='plotly_white',
    title=f"{n_grew} of {n_grew + n_shrank} countries grew between 2000 and 2022; of the {n_shrank} that shrank,<br>"
          f"{n_eu} are in Europe, mostly the Baltics and Eastern Europe",
)
fig.update_traces(marker_line_color='#888888', marker_line_width=0.3)
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_coloraxes(colorbar=dict(title='% change<br>2000–2022', thickness=12, len=0.6,
                                   tickvals=[-60, -30, 0, 30, 60],
                                   ticktext=['≤ −60', '−30', '0', '+30', '≥ +60']))
fig.add_annotation(text='Colour scale capped at ±60%; hover for exact values. Very small territories are too small to see.',
                   xref='paper', yref='paper', x=0, y=-0.02, showarrow=False, font=dict(size=10, color='#777777'))
fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=90, b=30, l=0, r=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** **Treemap.** The question asks for a part-to-whole view with two nested levels (continent, then country), which is exactly what a treemap does: tile area is proportional to 2022 population and the hierarchy is visible in one figure. Colour encodes continent (same palette as the other figures), so the country tiles inherit their continent's colour and no legend is needed.

In [9]:
# Q6
CONT_COLORS = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
               'North America': '#E69F00', 'South America': '#56B4E9', 'Oceania': '#666666'}

total = df['2022 Population'].sum()
asia_share = df.loc[df['Continent'] == 'Asia', '2022 Population'].sum() / total
ci_share = df.loc[df['Country'].isin(['China', 'India']), '2022 Population'].sum() / total

fig = px.treemap(
    df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
    color='Continent', color_discrete_map={**CONT_COLORS, '(?)': '#FFFFFF'}, template='plotly_white',
    title=f"Asia holds {asia_share:.0%} of the world's {total / 1e9:.1f} billion people;<br>"
          f"China and India alone account for {ci_share:.0%}",
)
fig.update_traces(root_color='white', marker_line_color='white', marker_line_width=1.5,
                  texttemplate='%{label}<br>%{percentRoot:.1%}', textfont_size=13)
fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=90, l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** **100% stacked bar chart.** The question is about *shares* of a whole that always sums to 100%, so a 100% stacked bar fits best: the absolute total changes (3.7 to 8 billion) but only the composition matters here. Africa (rising) is the accent and sits at the bottom, Europe (falling) sits at the top, and each has a flat baseline, so their change is easy to read; Asia and the rest are grey context, and direct labels replace the legend. Years are a categorical axis, so bars are evenly spaced even though the gaps between years are not.

In [10]:
# Q7
d = df_long.copy()
d['Group'] = np.where(d['Continent'].isin(['Africa', 'Asia', 'Europe']), d['Continent'], 'Rest of the world')
shares = d.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
shares['Share'] = shares['Population'] / shares.groupby('Year')['Population'].transform('sum') * 100
shares['Year'] = shares['Year'].astype(str)

stack_order = ['Africa', 'Asia', 'Rest of the world', 'Europe']    # bottom -> top
colors = {'Africa': '#D55E00', 'Europe': '#0072B2', 'Asia': '#8C8C8C', 'Rest of the world': '#D9D9D9'}
s = shares.pivot(index='Year', columns='Group', values='Share')

fig = px.bar(
    shares, x='Year', y='Share', color='Group', text='Share',
    category_orders={'Group': stack_order, 'Year': sorted(shares['Year'].unique())},
    color_discrete_map=colors, template='plotly_white',
    title=f"Africa's share of world population rose from {s.loc['1970', 'Africa']:.0f}% to {s.loc['2022', 'Africa']:.0f}% "
          f"since 1970,<br>while Europe's fell from {s.loc['1970', 'Europe']:.0f}% to {s.loc['2022', 'Europe']:.0f}%",
)
fig.update_traces(texttemplate='%{y:.0f}%', textposition='inside', insidetextanchor='middle', marker_line_width=0)
fig.update_traces(textfont_color='white', selector=dict(name='Africa'))
fig.update_traces(textfont_color='white', selector=dict(name='Europe'))

# direct labels to the right of the last bar (x = 7 is the 2022 bar on the categorical axis)
bottom = 0
for g in stack_order:
    fig.add_annotation(x=7.55, y=bottom + s.loc['2022', g] / 2, text=g, showarrow=False, xanchor='left',
                       font=dict(size=12, color=colors[g] if g in ('Africa', 'Europe') else '#666666'))
    bottom += s.loc['2022', g]

fig.update_layout(showlegend=False, title_x=0, title_font_size=18, margin=dict(t=90, r=140),
                  xaxis_title=None, yaxis_title=None)
fig.update_xaxes(range=[-0.5, 8.5], showgrid=False)
fig.update_yaxes(visible=False, range=[0, 100])
fig.show()